In [1]:
"""
Cell 1: Setup & Load Data
--------------------------
Load the unified cleaned dataset and explore its structure.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("=" * 70)
print("DATASET REPLAY PREPARATION - Cell 1: Setup & Load")
print("=" * 70)

# Define paths
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()),
    Path.cwd(),
)
data_dir = project_root / 'data' / 'processed'

# Check available datasets
print("\n📂 Available datasets:")
if data_dir.exists():
    csv_files = list(data_dir.glob('*.csv'))
    for i, f in enumerate(csv_files, 1):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"   {i}. {f.name} ({size_mb:.1f} MB)")
else:
    print(f"   ❌ Directory not found: {data_dir}")

# Load unified dataset
print("\n📥 Loading unified dataset...")
dataset_path = data_dir / 'ids_intrusion_cleaned_final.csv'

if not dataset_path.exists():
    print(f"❌ File not found: {dataset_path}")
    print("\nTrying alternative locations...")
    # Try other possible locations
    alternatives = [
        data_dir / 'unified_binary_dataset.csv',
        data_dir / 'combined_dataset.csv',
        project_root / 'data' / 'unified_dataset.csv'
    ]
    for alt in alternatives:
        if alt.exists():
            dataset_path = alt
            print(f"✅ Found: {alt}")
            break
    else:
        print("❌ No dataset found. Please check file location.")
        dataset_path = None

if dataset_path:
    # Load dataset
    df = pd.read_csv(dataset_path)

    print(f"\n✅ Dataset loaded: {len(df):,} rows")

    # Display basic info
    print("\n" + "=" * 70)
    print("DATASET OVERVIEW")
    print("=" * 70)

    print(f"\n📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(
        f"💾 Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

    # Check for label column
    label_candidates = ['label', 'Label', 'attack_type', 'class']
    label_col = None
    for col in label_candidates:
        if col in df.columns:
            label_col = col
            break

    if label_col:
        print(f"\n🏷️  Label column: '{label_col}'")
        print("\n📈 Label distribution:")
        print(df[label_col].value_counts())
        print("\nPercentages:")
        print(df[label_col].value_counts(normalize=True).mul(100).round(2))
    else:
        print("\n⚠️  No label column found. Checking for 'benign'/'malicious' binary...")
        # Check if there's a binary column
        for col in df.columns:
            if df[col].dtype == 'object' and set(df[col].unique()).issubset({'benign', 'malicious', 'Benign', 'Malicious'}):
                label_col = col
                print(f"✅ Found binary label: '{label_col}'")
                print(df[label_col].value_counts())
                break

    print("\n" + "=" * 70)
    print("FIRST 5 ROWS")
    print("=" * 70)
    print(df.head())

    print("\n" + "=" * 70)
    print("COLUMN NAMES")
    print("=" * 70)
    print(f"Total columns: {len(df.columns)}")
    for i, col in enumerate(df.columns, 1):
        print(f"{i:3d}. {col}")

    print("\n✅ Cell 1 Complete!")
    print("\n📋 Next steps:")
    print("   1. Verify the label column name")
    print("   2. Check if 78 features are present")
    print("   3. Confirm malicious/benign split is reasonable")

else:
    df = None
    print("\n❌ Cannot proceed without dataset")

DATASET REPLAY PREPARATION - Cell 1: Setup & Load

📂 Available datasets:
   1. cic_unified_balanced.csv (2483.6 MB)
   2. cic_unified_dataset.csv (6385.0 MB)
   3. ids_intrusion_cleaned.csv (296.5 MB)
   4. ids_intrusion_cleaned_final.csv (257.4 MB)
   5. cic_unified_cleaned.csv (6290.7 MB)

📥 Loading unified dataset...

✅ Dataset loaded: 671,138 rows

DATASET OVERVIEW

📊 Shape: 671,138 rows × 79 columns
💾 Memory usage: 440.4 MB

🏷️  Label column: 'Label'

📈 Label distribution:
Label
Benign            577037
SSH-Bruteforce     94048
FTP-BruteForce        53
Name: count, dtype: int64

Percentages:
Label
Benign            85.98
SSH-Bruteforce    14.01
FTP-BruteForce     0.01
Name: proportion, dtype: float64

FIRST 5 ROWS
   Dst Port  Protocol  Flow Duration  Tot Fwd Pkts  Tot Bwd Pkts  \
0         0         0      112641719             3             0   
1         0         0      112641466             3             0   
2         0         0      112638623             3             0   

In [2]:
"""
Cell 2: Sample Diverse Attacks & Create Balanced Replay Dataset
-----------------------------------------------------------------
Create a balanced dataset with good attack/benign ratio for realistic demo.
Target: 150-200 flows with temporal patterns simulating APT scenarios.
"""

print("=" * 70)
print("DATASET REPLAY PREPARATION - Cell 2: Sample Attacks")
print("=" * 70)

# Define sampling strategy
print("\n📋 Sampling Strategy:")
print("   Target: ~180 flows total")
print("   Mix: 60% malicious (108), 40% benign (72)")
print("   Attack diversity: Multiple attack types")
print("   Temporal: Grouped by 'attacker IP' to simulate APT campaigns")

# Sample malicious flows
print("\n🔴 Sampling malicious flows...")

malicious_samples = []

# 1. SSH Bruteforce - Primary attack type (60 samples)
ssh_samples = df[df['Label'] == 'SSH-Bruteforce'].sample(n=60, random_state=42)
malicious_samples.append(ssh_samples)
print(f"   ✅ SSH-Bruteforce: {len(ssh_samples)} samples")

# 2. FTP Bruteforce - All available (53 samples)
ftp_samples = df[df['Label'] == 'FTP-BruteForce']
malicious_samples.append(ftp_samples)
print(f"   ✅ FTP-BruteForce: {len(ftp_samples)} samples")

# Combine all malicious
malicious_df = pd.concat(malicious_samples, ignore_index=True)
print(f"\n   📊 Total malicious: {len(malicious_df)} flows")

# Sample benign flows
print("\n⚪ Sampling benign flows...")
benign_df = df[df['Label'] == 'Benign'].sample(n=72, random_state=42)
print(f"   ✅ Benign: {len(benign_df)} flows")

# Combine into replay dataset
print("\n🔄 Combining into replay dataset...")
replay_df = pd.concat([malicious_df, benign_df], ignore_index=True)

# Shuffle to mix attacks and benign traffic
replay_df = replay_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Replay dataset created: {len(replay_df)} flows")

# Display distribution
print("\n" + "=" * 70)
print("REPLAY DATASET DISTRIBUTION")
print("=" * 70)
print("\n📊 Label distribution:")
print(replay_df['Label'].value_counts())
print("\nPercentages:")
print(replay_df['Label'].value_counts(normalize=True).mul(100).round(2))

# Add temporal information for APT simulation
print("\n" + "=" * 70)
print("ADDING TEMPORAL PATTERNS")
print("=" * 70)

print("\n🕒 Simulating multi-stage APT scenarios...")

# Create synthetic source IPs to group attacks
# This simulates attacks from same adversary over time
np.random.seed(42)


def assign_source_ip(row):
    """Assign source IP to simulate APT campaigns."""
    if row['Label'] == 'Benign':
        # Benign traffic from random IPs
        return f"192.168.1.{np.random.randint(1, 255)}"
    elif row['Label'] == 'SSH-Bruteforce':
        # SSH attacks from 3 different adversary IPs
        attacker_id = np.random.choice([100, 101, 102])
        return f"10.0.0.{attacker_id}"
    elif row['Label'] == 'FTP-BruteForce':
        # FTP attacks from 2 different adversary IPs
        attacker_id = np.random.choice([103, 104])
        return f"10.0.0.{attacker_id}"
    else:
        return "127.0.0.1"


replay_df['src_ip'] = replay_df.apply(assign_source_ip, axis=1)

# Assign destination IPs (internal network)
replay_df['dst_ip'] = replay_df.apply(
    lambda x: f"172.16.0.{np.random.randint(10, 50)}" if x['Label'] != 'Benign'
    else f"172.16.0.{np.random.randint(1, 255)}",
    axis=1
)

# Add timestamps (sequential with realistic spacing)
# Start at specific time and add 2-5 seconds between flows
base_time = pd.Timestamp('2024-01-15 14:00:00')
time_increments = np.random.randint(2, 6, size=len(replay_df))
timestamps = [base_time + pd.Timedelta(seconds=int(sum(time_increments[:i])))
              for i in range(len(replay_df))]
replay_df['timestamp'] = timestamps

print(
    f"   ✅ Source IPs assigned (simulating {replay_df['src_ip'].nunique()} unique sources)")
print("   ✅ Destination IPs assigned")
print(
    f"   ✅ Timestamps added (spanning {(timestamps[-1] - timestamps[0]).seconds / 60:.1f} minutes)")

# Group by attacker to see APT patterns
print("\n📊 Attack patterns by source IP:")
attack_patterns = replay_df[replay_df['Label'] !=
                            'Benign'].groupby(['src_ip', 'Label']).size()
for (ip, label), count in attack_patterns.items():
    print(f"   • {ip}: {count} {label} flows")

# Check feature completeness
print("\n" + "=" * 70)
print("FEATURE VALIDATION")
print("=" * 70)

# Verify all 78 features are present (excluding Label, src_ip, dst_ip, timestamp)
feature_cols = [col for col in replay_df.columns if col not in [
    'Label', 'src_ip', 'dst_ip', 'timestamp']]
print(f"\n✅ Features present: {len(feature_cols)}")
print("✅ Required for ML: 78")
print(f"{'✅ VALID' if len(feature_cols) >= 78 else '❌ MISSING FEATURES'}")

# Check for missing values
missing = replay_df[feature_cols].isnull().sum().sum()
print(f"\n✅ Missing values: {missing}")

# Check for infinite values
inf_count = np.isinf(replay_df[feature_cols].select_dtypes(
    include=[np.number])).sum().sum()
print(f"✅ Infinite values: {inf_count}")

print("\n" + "=" * 70)
print("SAMPLE PREVIEW")
print("=" * 70)
print("\n🔍 First 10 flows with metadata:")
preview_cols = ['timestamp', 'src_ip', 'dst_ip',
                'Dst Port', 'Protocol', 'Flow Duration', 'Label']
print(replay_df[preview_cols].head(10).to_string())

print("\n✅ Cell 2 Complete!")
print("\n📋 Next steps:")
print("   1. Add more attack diversity if needed")
print("   2. Validate temporal patterns")
print("   3. Save replay dataset to file")

DATASET REPLAY PREPARATION - Cell 2: Sample Attacks

📋 Sampling Strategy:
   Target: ~180 flows total
   Mix: 60% malicious (108), 40% benign (72)
   Attack diversity: Multiple attack types
   Temporal: Grouped by 'attacker IP' to simulate APT campaigns

🔴 Sampling malicious flows...
   ✅ SSH-Bruteforce: 60 samples
   ✅ FTP-BruteForce: 53 samples

   📊 Total malicious: 113 flows

⚪ Sampling benign flows...
   ✅ Benign: 72 flows

🔄 Combining into replay dataset...

✅ Replay dataset created: 185 flows

REPLAY DATASET DISTRIBUTION

📊 Label distribution:
Label
Benign            72
SSH-Bruteforce    60
FTP-BruteForce    53
Name: count, dtype: int64

Percentages:
Label
Benign            38.92
SSH-Bruteforce    32.43
FTP-BruteForce    28.65
Name: proportion, dtype: float64

ADDING TEMPORAL PATTERNS

🕒 Simulating multi-stage APT scenarios...
   ✅ Source IPs assigned (simulating 67 unique sources)
   ✅ Destination IPs assigned
   ✅ Timestamps added (spanning 10.7 minutes)

📊 Attack patterns by 

In [3]:
"""
Cell 3: Save Replay Dataset & Create Metadata
----------------------------------------------
Save the prepared dataset for streaming replay and create accompanying metadata.
"""

import json  # noqa: E402
print("=" * 70)
print("DATASET REPLAY PREPARATION - Cell 3: Save & Export")
print("=" * 70)

# Define output paths
output_dir = project_root / 'data' / 'replay'
output_dir.mkdir(parents=True, exist_ok=True)

dataset_path = output_dir / 'apt_scenario_1.csv'
metadata_path = output_dir / 'apt_scenario_1_metadata.json'

print(f"\n📂 Output directory: {output_dir}")

# Prepare final dataset for export
print("\n📋 Preparing dataset for export...")

# Select columns for replay (78 features + metadata)
# We'll keep: all 78 features + Label + src_ip + dst_ip + timestamp
export_df = replay_df.copy()

# Reorder columns: metadata first, then features, then label
metadata_cols = ['timestamp', 'src_ip', 'dst_ip']
feature_cols = [col for col in export_df.columns
                if col not in ['Label', 'timestamp', 'src_ip', 'dst_ip']]
final_cols = metadata_cols + feature_cols + ['Label']

export_df = export_df[final_cols]

print(
    f"   ✅ Columns ordered: {len(metadata_cols)} metadata + {len(feature_cols)} features + 1 label")

# Save to CSV
print("\n💾 Saving replay dataset...")
export_df.to_csv(dataset_path, index=False)
file_size_mb = dataset_path.stat().st_size / (1024 * 1024)
print(f"   ✅ Saved: {dataset_path}")
print(f"   📦 Size: {file_size_mb:.2f} MB")

# Create metadata file
print("\n📝 Creating metadata...")

metadata = {
    "dataset_info": {
        "name": "APT Scenario 1 - Brute Force Campaign",
        "description": "Simulated APT campaign with SSH and FTP brute force attacks from multiple adversaries",
        "created_at": pd.Timestamp.now().isoformat(),
        "source_dataset": "cic_unified_balanced.csv",
        "total_flows": len(export_df),
        "duration_minutes": (export_df['timestamp'].max() - export_df['timestamp'].min()).seconds / 60,
        "time_range": {
            "start": export_df['timestamp'].min().isoformat(),
            "end": export_df['timestamp'].max().isoformat()
        }
    },
    "label_distribution": {
        "total": len(export_df),
        "benign": int(export_df[export_df['Label'] == 'Benign'].shape[0]),
        "malicious": int(export_df[export_df['Label'] != 'Benign'].shape[0]),
        "attack_types": export_df[export_df['Label'] != 'Benign']['Label'].value_counts().to_dict()
    },
    "attack_patterns": {
        "unique_attackers": int(export_df[export_df['Label'] != 'Benign']['src_ip'].nunique()),
        "unique_targets": int(export_df[export_df['Label'] != 'Benign']['dst_ip'].nunique()),
        "attacker_profiles": {}
    },
    "features": {
        "count": len(feature_cols),
        "columns": feature_cols
    },
    "usage": {
        "replay_rate": "Real-time (2-5 seconds between flows)",
        "recommended_duration": "2-3 minutes for full replay",
        "detection_expectations": {
            "ml_should_detect": "SSH-Bruteforce and FTP-BruteForce with high confidence",
            "llm_should_analyze": "Attack progression patterns and multi-stage campaigns",
            "memory_should_correlate": "Repeat attacks from same source IPs"
        }
    }
}

# Add detailed attacker profiles
for ip in export_df[export_df['Label'] != 'Benign']['src_ip'].unique():
    attacker_flows = export_df[export_df['src_ip'] == ip]
    metadata["attack_patterns"]["attacker_profiles"][ip] = {
        "attack_count": int(len(attacker_flows)),
        "attack_types": attacker_flows['Label'].value_counts().to_dict(),
        "first_seen": attacker_flows['timestamp'].min().isoformat(),
        "last_seen": attacker_flows['timestamp'].max().isoformat(),
        "duration_seconds": int((attacker_flows['timestamp'].max() - attacker_flows['timestamp'].min()).seconds),
        "targeted_hosts": attacker_flows['dst_ip'].nunique()
    }

# Save metadata
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"   ✅ Saved: {metadata_path}")

# Display metadata summary
print("\n" + "=" * 70)
print("METADATA SUMMARY")
print("=" * 70)

print("\n📊 Dataset Overview:")
print(f"   • Total flows: {metadata['dataset_info']['total_flows']}")
print(
    f"   • Time span: {metadata['dataset_info']['duration_minutes']:.1f} minutes")
print(f"   • Benign: {metadata['label_distribution']['benign']} ({metadata['label_distribution']['benign']/metadata['dataset_info']['total_flows']*100:.1f}%)")
print(f"   • Malicious: {metadata['label_distribution']['malicious']} ({metadata['label_distribution']['malicious']/metadata['dataset_info']['total_flows']*100:.1f}%)")

print("\n🎯 Attack Types:")
for attack_type, count in metadata['label_distribution']['attack_types'].items():
    print(f"   • {attack_type}: {count} flows")

print("\n👥 Attacker Profiles:")
for ip, profile in metadata['attack_patterns']['attacker_profiles'].items():
    print(f"   • {ip}:")
    print(f"     - Attack count: {profile['attack_count']}")
    print(f"     - Duration: {profile['duration_seconds']}s")
    print(f"     - Targets: {profile['targeted_hosts']} hosts")
    attack_types_str = ', '.join(
        [f"{k}({v})" for k, v in profile['attack_types'].items()])
    print(f"     - Types: {attack_types_str}")

# Create a quick reference guide
print("\n" + "=" * 70)
print("USAGE GUIDE")
print("=" * 70)

usage_guide = f"""
📖 How to Use This Replay Dataset
──────────────────────────────────

1. **File Location:**
   {dataset_path}

2. **Replay Script Usage:**
   # Load the dataset
   df = pd.read_csv('{dataset_path}')
   
   # Stream flows in order (respecting timestamps)
   for idx, row in df.iterrows():
       # Extract features (columns 3:-1)
       features = row[3:-1].values
       label = row['Label']
       
       # Process through detection system
       detect_flow(features, label)
       
       # Wait for next flow (simulate real-time)
       if idx < len(df) - 1:
           time_diff = (df.iloc[idx+1]['timestamp'] - row['timestamp']).seconds
           time.sleep(time_diff)

3. **Expected System Behavior:**
   ✅ ML Models: Should detect SSH/FTP brute force with >90% confidence
   ✅ LLM Analysis: Should identify attack progression patterns
   ✅ Memory System: Should correlate 5 distinct attackers
   ✅ Dashboard: Should show real-time threat updates

4. **Key Scenarios to Highlight:**
   • 10.0.0.100: 22 SSH brute force attempts (sustained campaign)
   • 10.0.0.103: 28 FTP brute force attempts (aggressive attacker)
   • Mixed benign traffic providing realistic noise floor

5. **Validation Checklist:**
   ✅ {len(export_df)} flows loaded
   ✅ 78 features per flow
   ✅ {metadata['attack_patterns']['unique_attackers']} unique attackers
   ✅ Time-ordered from {metadata['dataset_info']['time_range']['start']} to {metadata['dataset_info']['time_range']['end']}
"""

print(usage_guide)

# Save usage guide
guide_path = output_dir / 'USAGE_GUIDE.txt'
with open(guide_path, 'w') as f:
    f.write(usage_guide)
print(f"💾 Usage guide saved: {guide_path}")

print("\n" + "=" * 70)
print("✅ CELL 3 COMPLETE - DATASET READY FOR REPLAY")
print("=" * 70)

print("\n📦 Output Files:")
print(f"   1. Dataset: {dataset_path.name}")
print(f"   2. Metadata: {metadata_path.name}")
print(f"   3. Guide: {guide_path.name}")

print("\n🚀 Next Step: Create streaming replay detector")
print("   Location: src/network/dataset_replay_detector.py")

DATASET REPLAY PREPARATION - Cell 3: Save & Export

📂 Output directory: ./data/replay

📋 Preparing dataset for export...
   ✅ Columns ordered: 3 metadata + 78 features + 1 label

💾 Saving replay dataset...
   ✅ Saved: ./data/replay/apt_scenario_1.csv
   📦 Size: 0.08 MB

📝 Creating metadata...
   ✅ Saved: ./data/replay/apt_scenario_1_metadata.json

METADATA SUMMARY

📊 Dataset Overview:
   • Total flows: 185
   • Time span: 10.7 minutes
   • Benign: 72 (38.9%)
   • Malicious: 113 (61.1%)

🎯 Attack Types:
   • SSH-Bruteforce: 60 flows
   • FTP-BruteForce: 53 flows

👥 Attacker Profiles:
   • 10.0.0.102:
     - Attack count: 18
     - Duration: 582s
     - Targets: 15 hosts
     - Types: SSH-Bruteforce(18)
   • 10.0.0.100:
     - Attack count: 22
     - Duration: 618s
     - Targets: 16 hosts
     - Types: SSH-Bruteforce(22)
   • 10.0.0.103:
     - Attack count: 28
     - Duration: 631s
     - Targets: 22 hosts
     - Types: FTP-BruteForce(28)
   • 10.0.0.101:
     - Attack count: 20
     -